# Decision Analysis 6 6


## Electre Tri-B


The objective of this laboratory session is to gain practical understanding of the **ELECTRE TRI-B** method – one of the most important _outranking-based sorting_ methods used in Multi-Criteria Decision Analysis (MCDA).

The ELECTRE TRI-B method is designed to assign a finite set of decision alternatives to predefined, ordered categories based on their performance with respect to multiple, often conflicting criteria. Instead of producing a complete ranking, the method focuses on classification by comparing alternatives with reference profiles that define the boundaries between categories.

Within the method, each alternative is compared to a set of boundary profiles that separate adjacent categories. For each criterion, the difference between the performance of the alternative and the profile is evaluated. This comparison is then used to construct a **concordance index**, which expresses the degree to which there is sufficient evidence to support the assertion that the alternative is at least as good as the profile.

To account for strong opposition on individual criteria, **discordance indices** are also computed. These indices capture situations where a significant disadvantage on a single criterion may weaken or even invalidate the overall outranking relation.

The concordance and discordance information are aggregated into a **credibility index**, which represents the strength of the outranking relation between the alternative and the profile. This index is then compared with a predefined cutting level ($\lambda$) to determine whether the outranking relation is validated.

The assignment procedure is performed using one of two possible rules:

- **pessimistic (conjunctive) assignment** – the alternative is assigned to the highest category for which it sufficiently outranks the lower boundary profile,
- **optimistic (disjunctive) assignment** – the alternative is assigned to the lowest category whose upper boundary profile does not sufficiently outrank the alternative.

In the ELECTRE TRI-B method, the decision maker provides preference information in the following form:

- **q** – indifference threshold,
- **p** – preference threshold,
- **v** – veto threshold,
- **w** – weight of criterion _k_,
- **$\lambda$** – cutting level for validating the outranking relation,
- direction of preference (cost or benefit criterion).

During the laboratory sessions, we will consider a simplified version of the ELECTRE method, in which a single set of preference information is defined instead of a separate set for each profile.

> **IMPORTANT**
>
> Code written in this notebook will be checked against automatic code checker and the points will be given based on its' results, please leave the function signatures unchanged.
> As a result there are no partial points for a programing tasks


In [180]:
from pathlib import Path

import numpy as np
import pandas as pd

import utils

In [181]:
import pandas as pd
import numpy as np

# Load dataset
dataset = pd.read_csv("dataset.csv", index_col=0)
print("Dataset:")
print(dataset)
print()

boundary_profiles = pd.read_csv("boundary_profiles.csv", index_col=0)
boundary_profiles.index.name = "Boundary"

# Per-profile preference information
# b1 thresholds - less strict (lower boundary)
preference_information = pd.read_csv("preference.csv", index_col=0)
preference_information.index.name = "Criterion"

# Credibility threshold
credibility_threshold = 0.70

print("Boundary profiles:")
print(boundary_profiles)
print()
print("Preference information b1:")
print(preference_information)
print()
print(f"Credibility threshold: {credibility_threshold}")
print()
print("Setup saved successfully!")

Dataset:
            Hours_Studied  Attendance  Sleep_Hours  Previous_Scores  \
Student_1              23          84            7               73   
Student_2              19          64            8               59   
Student_3              24          98            7               91   
Student_4              29          89            8               98   
Student_5              19          92            6               65   
Student_6              19          88            8               89   
Student_7              29          84            7               68   
Student_8              25          78            6               50   
Student_9              17          94            6               80   
Student_10             23          98            8               71   
Student_11             17          80            8               88   
Student_12             17          97            6               87   
Student_13             21          83            8               97 

In [182]:
dataset

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions
Student_1,23,84,7,73,0
Student_2,19,64,8,59,2
Student_3,24,98,7,91,2
Student_4,29,89,8,98,1
Student_5,19,92,6,65,3
Student_6,19,88,8,89,3
Student_7,29,84,7,68,1
Student_8,25,78,6,50,1
Student_9,17,94,6,80,0
Student_10,23,98,8,71,0


In [183]:
boundary_profiles

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions
Boundary,,,,,
b1,15,78,7,72,1
b2,22,88,8,86,3


In [184]:
preference_information

,q,p,v,w,type
Criterion,,,,,
Hours_Studied,1,4,15.0,5,gain
Attendance,2,8,20.0,4,gain
Sleep_Hours,0,1,NaN,1,gain
Previous_Scores,3,10,20.0,2,gain
Tutoring_Sessions,0,1,NaN,3,gain


### Task 1 (maximum points: 1)

Implement the `difference_function` function.

$$
    d_j (a, b) = \left\{ \begin{array}{ll}
    g_j(a) - g_j(b), & \textrm{for \textit{gain} criterion},\\
    g_j(b) - g_j(a), & \textrm{for \textit{cost} criterion},\\
    \end{array} \right.
$$

This difference is used later to compute marginal concordance and discordance values.


In [185]:
def difference_function(
    alternative_a: float, alternative_b: float, criterion_type: utils.CriterionType
) -> float:
    """
    Function that calculates the difference between given alternative pair on a single criterion.

    :param alternative_a: first alternative in a pair
    :param alternative_b: second alternative in a pair
    :param criterion_type: criterion type either gain or cost
    :return: difference between alternative pair calculated according to the criterion type
    """
    if criterion_type == utils.CriterionType.GAIN:
        return alternative_a - alternative_b
    elif criterion_type == utils.CriterionType.COST:
        return alternative_b - alternative_a
    else:
        raise ValueError("Invalid criterion type")

### Task 2 (maximum points: 1)

Implement the `calculate_marginal_concordance_index` function.

Requirements:

- Use inputs `diff`, `indifference_threshold` ($q$), and `preference_threshold` ($p$).
- Compute the marginal concordance value with the piecewise rule:

$$
c(a, b)=\begin{cases}
1, & d(a,b) \ge -q \\
0, & d(a,b) \le -p \\
\frac{p + d(a,b)}{p - q}, & -q > d(a,b) > -p
\end{cases}
$$


In [186]:
def calculate_marginal_concordance_index(
    diff: float, indifference_threshold: float, preference_threshold: float
) -> float:
    """
    Function that calculates the marginal concordance index for the given pair of alternatives, according to the formula presented during classes.

    :param diff: difference between compared alternatives either as a float for single criterion and alternative pairs, or as numpy array for multiple alternatives
    :param indifference_threshold: indifference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :param preference_threshold: preference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :return: marginal concordance index either as a float for single criterion and alternative pairs, or as numpy array for multiple criterion
    """
    if diff >= -indifference_threshold:
        return 1.0
    elif diff <= -preference_threshold:
        return 0.0
    else:
        return (preference_threshold + diff) / (
            preference_threshold - indifference_threshold
        )

### Task 3 (maximum points: 1)

Implement the `calculate_marginal_concordance_matrix` function.

Requirements:

- Use `dataset`, `boundary_profiles`, and `preference_information`.
- Build differences for both directions: alternative vs profile and profile vs alternative.
- Use `q` and `p` thresholds from `preference_information`.
- Return a 4D matrix with shape `[2, n_alternatives, n_profiles, n_criteria]`.


In [187]:
def calculate_marginal_concordance_matrix(
    dataset: pd.DataFrame,
    boundary_profiles: pd.DataFrame,
    preference_information: pd.DataFrame,
) -> np.ndarray:
    """
    Function that calculates the marginal concordance matrix for all alternatives pairs and criterion available in dataset

    :param dataset: pandas dataframe representing dataset with alternatives as rows and criterion as columns
    :param boundary_profiles: pandas dataframe with boundary profiles
    :param preference_information: pandas dataframe with preference information for all criterion
    :return: 4D numpy array with marginal concordance matrix with shape [2, number of alternatives, number of boundary profiles, number of criterion], where element with index [0, i, j, k] describe marginal concordance index between alternative i and boundary profile j on criterion k, while element with index [1, i, j, k] describe marginal concordance index between boundary profile j and  alternative i on criterion k
    """
    num_alternatives = dataset.shape[0]
    num_boundary_profiles = boundary_profiles.shape[0]
    num_criteria = dataset.shape[1]

    marginal_concordance_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles, num_criteria)
    )
    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            for k in range(num_criteria):
                c_name = dataset.columns[k]
                c_type = preference_information.loc[
                    preference_information.index == c_name, "type"
                ].values[0]
                diff_a_b = difference_function(
                    dataset.iloc[i, k], boundary_profiles.iloc[j, k], c_type
                )
                diff_b_a = difference_function(
                    boundary_profiles.iloc[j, k], dataset.iloc[i, k], c_type
                )
                indifference_threshold = preference_information.loc[
                    preference_information.index == c_name,
                    "q",
                ].values[0]
                preference_threshold = preference_information.loc[
                    preference_information.index == c_name,
                    "p",
                ].values[0]
                marginal_concordance_matrix[0, i, j, k] = (
                    calculate_marginal_concordance_index(
                        diff_a_b, indifference_threshold, preference_threshold
                    )
                )
                marginal_concordance_matrix[1, i, j, k] = (
                    calculate_marginal_concordance_index(
                        diff_b_a, indifference_threshold, preference_threshold
                    )
                )

    return np.nan_to_num(marginal_concordance_matrix, nan=0.0)

In [188]:
marginal_concordance_matrix = calculate_marginal_concordance_matrix(
    dataset, boundary_profiles, preference_information
)

In [189]:
marginal_concordance_matrix[:, 4, 0]

array([[1.        , 1.        , 0.        , 0.42857143, 1.        ],
       [0.        , 0.        , 1.        , 1.        , 0.        ]])

### Task 4 (maximum points: 1)

Implement the `calculate_comprehensive_concordance_matrix` function.

The function should aggregate marginal concordance values across all criteria using criterion weights.

For each pair of alternatives and boundary profile (a, b), compute the comprehensive concordance index:

$$C(a, b) = \frac{\sum w_k * c_k(a, b)}{\sum w_k}$$

where:

- $w_k$ is the weight of criterion k,
- $c_k(a, b)$ is the marginal concordance value.


In [190]:
def calculate_comprehensive_concordance_matrix(
    marginal_concordance_matrix: np.ndarray, preference_information: pd.DataFrame
) -> np.ndarray:
    num_alternatives = marginal_concordance_matrix.shape[1]
    num_boundary_profiles = marginal_concordance_matrix.shape[2]
    num_criteria = marginal_concordance_matrix.shape[3]

    comprehensive_concordance_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles)
    )

    total_weight = sum(
        preference_information.loc[
            preference_information.index == dataset.columns[k], "w"
        ].values[0]
        for k in range(num_criteria)
    )

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            concordance_a_b = 0.0
            concordance_b_a = 0.0
            for k in range(num_criteria):
                c_name = dataset.columns[k]
                weight = preference_information.loc[
                    preference_information.index == c_name, "w"
                ].values[0]
                concordance_a_b += weight * marginal_concordance_matrix[0, i, j, k]
                concordance_b_a += weight * marginal_concordance_matrix[1, i, j, k]

            # Divide by sum of weights
            comprehensive_concordance_matrix[0, i, j] = concordance_a_b / total_weight
            comprehensive_concordance_matrix[1, i, j] = concordance_b_a / total_weight

    return comprehensive_concordance_matrix

In [191]:
comprehensive_concordance_index = calculate_comprehensive_concordance_matrix(
    marginal_concordance_matrix, preference_information
)

In [192]:
print(comprehensive_concordance_index[0, 4, 1])  # C(a_i, b_i)
print(comprehensive_concordance_index[1, 4, 0])  # C(b_i, a_i)

0.5777777777777777
0.2


### Task 5 (maximum points: 1)

Implement the `calculate_marginal_discordance_index` function.

Requirements:

- Use inputs `diff`, `ppreference_threshold` (p), and `veto threshold` (v).
- Compute the marginal discordance value with the piecewise rule:

$$
    D_j (a_i, b_{h}) = \left\{ \begin{array}{ll}
        1, & \textrm{if } d_k(a_{i}, b_{h}) \leq -v_j^h),\\
        0, & \textrm{if } d_k(a_{i}, b_{h}) \geq -p_j^h),\\
        \frac{-d_k(a_{i}, b_{h}) - p_j^h}{v_j^h - p_j^h}, & \textrm{if } -p_j^h > d_k(a_{i}, b_{h}) > -v_j^h).
    \end{array} \right.
$$


In [193]:
def calculate_marginal_discordance_index(
    diff: float, preference_threshold: float, veto_threshold: float
) -> float:
    """
    Function that calculates the marginal discordance index for the given pair of alternatives, according to the formula presented during classes.

    :param diff: difference between compared alternatives either as a float for single criterion and alternative pairs, or as numpy array for multiple alternatives
    :param preference_threshold: preference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :param veto_threshold: veto threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :return: marginal discordance index either as a float for single criterion and alternative pairs, or as numpy array for multiple criterion
    """
    if diff <= -veto_threshold:
        return 1.0
    elif diff >= -preference_threshold:
        return 0.0
    else:
        return (-diff - preference_threshold) / (veto_threshold - preference_threshold)

### Task 6 (maximum points: 1)

Implement the `calculate_marginal_discordance_matrix` function.

Requirements:

- Use `dataset`, `boundary_profiles`, `preference_thresholds`, `veto_thresholds`, and `criterion_types`.
- Build differences for both directions: alternative vs profile and profile vs alternative.
- Compute discordance values using `calculate_marginal_discordance_index`.
- Return a 4D matrix with shape `[2, n_alternatives, n_profiles, n_criteria]`.


In [194]:
def calculate_marginal_discordance_matrix(
    dataset: pd.DataFrame,
    boundary_profiles: pd.DataFrame,
    preference_thresholds,
    veto_thresholds,
    criterion_types,
) -> np.ndarray:
    """
    Function that calculates the marginal discordance matrix for all alternatives pairs and criterion available in dataset

    :param dataset: pandas dataframe representing dataset with alternatives as rows and criterion as columns
    :param boundary_profiles: pandas dataframe with boundary profiles
    :param preference_thresholds: pandas dataframe representing preference thresholds for all boundary profiles and criterion
    :param veto_thresholds: pandas dataframe representing veto thresholds for all boundary profiles and criterion
    :param criterion_types: pandas dataframe with a column 'type' representing the type of criterion (either gain or cost)
    :return: 4D numpy array with marginal discordance matrix with shape [2, number of alternatives, number of boundary profiles, number of criterion], where element with index [0, i, j, k] describe marginal discordance index between alternative i and boundary profile j on criterion k, while element with index [1, i, j, k] describe marginal discordance index between boundary profile j and  alternative i on criterion k
    """
    num_alternatives = dataset.shape[0]
    num_boundary_profiles = boundary_profiles.shape[0]
    num_criteria = dataset.shape[1]

    marginal_discordance_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles, num_criteria)
    )

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            for k in range(num_criteria):
                c_name = dataset.columns[k]
                c_type = criterion_types.loc[
                    criterion_types.index == c_name, "type"
                ].values[0]
                preference_threshold = preference_thresholds.loc[
                    preference_thresholds.index == c_name, "p"
                ].values[0]
                veto_threshold = veto_thresholds.loc[
                    veto_thresholds.index == c_name, "v"
                ].values[0]

                diff_a_b = difference_function(
                    dataset.iloc[i, k], boundary_profiles.iloc[j, k], c_type
                )
                diff_b_a = difference_function(
                    boundary_profiles.iloc[j, k], dataset.iloc[i, k], c_type
                )

                marginal_discordance_matrix[0, i, j, k] = (
                    calculate_marginal_discordance_index(
                        diff_a_b, preference_threshold, veto_threshold
                    )
                )
                marginal_discordance_matrix[1, i, j, k] = (
                    calculate_marginal_discordance_index(
                        diff_b_a, preference_threshold, veto_threshold
                    )
                )

    return np.nan_to_num(marginal_discordance_matrix, nan=0.0)

### Task 7 (maximum points: 1)

Implement the `calculate_credibility_index` function.

The function should compute the credibility index by combining the comprehensive concordance matrix with marginal discordance indices.

For each pair of alternatives and boundary profile (a, b), compute the credibility index:

$$\sigma(a, b) = C(a, b) \cdot \prod_{d_k(a, b) > C(a, b)} \frac{1 - d_k(a, b)}{1 - C(a, b)}$$

where:

- $C(a, b)$ is the comprehensive concordance index,
- $d_k(a, b)$ is the marginal discordance index on criterion k,
- the product is taken over criteria where discordance exceeds concordance.


In [195]:
def calculate_credibility_index(
    comprehensive_concordance_matrix: np.ndarray,
    marginal_discordance_matrix: np.ndarray,
) -> np.ndarray:
    """
    Function that calculates the credibility index for the given comprehensive concordance matrix and marginal discordance matrix

    :param comprehensive_concordance_matrix: 3D numpy array with comprehensive concordance matrix. Every entry in the matrix [i, j] represents comprehensive concordance index between alternative i and alternative j
    :param marginal_discordance_matrix: 3D numpy array with marginal discordance matrix, Consecutive indices [i, j, k] describe first alternative, second alternative, criterion
    :return: 3D numpy array with credibility matrix with shape [2, number of alternatives, number of boundary profiles], where element with index [0, i, j] describe credibility index between alternative i and boundary profile j, while element with index [1, i, j] describe credibility index between boundary profile j and  alternative i
    """
    num_alternatives = comprehensive_concordance_matrix.shape[1]
    num_boundary_profiles = comprehensive_concordance_matrix.shape[2]
    num_criteria = marginal_discordance_matrix.shape[3]

    credibility_matrix = np.zeros((2, num_alternatives, num_boundary_profiles))

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            credibility_a_b = comprehensive_concordance_matrix[0, i, j]
            credibility_b_a = comprehensive_concordance_matrix[1, i, j]
            for k in range(num_criteria):
                if marginal_discordance_matrix[0, i, j, k] > credibility_a_b:
                    credibility_a_b *= (1 - marginal_discordance_matrix[0, i, j, k]) / (
                        1 - comprehensive_concordance_matrix[0, i, j]
                    )
                if marginal_discordance_matrix[1, i, j, k] > credibility_b_a:
                    credibility_b_a *= (1 - marginal_discordance_matrix[1, i, j, k]) / (
                        1 - comprehensive_concordance_matrix[1, i, j]
                    )
            credibility_matrix[0, i, j] = credibility_a_b
            credibility_matrix[1, i, j] = credibility_b_a

    return credibility_matrix

### Task 8 (maximum points: 1)

Implement the `calculate_outranking_relation_matrix` function.

The function should compare credibility values with the cutting level $\lambda$ (`credibility_threshold`) and return a boolean matrix indicating whether outranking holds for each pair.


In [196]:
def calculate_outranking_relation_matrix(
    credibility_index: np.ndarray, credibility_threshold: float
) -> np.ndarray:
    """
    Function that calculates boolean matrix with information if outranking holds for a given pair

    :param credibility_index: 3D numpy array with credibility matrix with shape [2, number of alternatives, number of boundary profiles], where element with index [0, i, j] describe credibility index between alternative i and boundary profile j, while element with index [1, i, j] describe credibility index between boundary profile j and  alternative i
    :param credibility_threshold: float number
    :return: 3D numpy boolean matrix with information if outranking holds for a given pair
    """
    num_alternatives = credibility_index.shape[1]
    num_boundary_profiles = credibility_index.shape[2]

    outranking_relation_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles), dtype=bool
    )

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            outranking_relation_matrix[0, i, j] = (
                credibility_index[0, i, j] >= credibility_threshold
            )
            outranking_relation_matrix[1, i, j] = (
                credibility_index[1, i, j] >= credibility_threshold
            )

    return outranking_relation_matrix

### Task 9 (maximum points: 1)

Implement the `calculate_pessimistic_assigment` function.

Use the pessimistic rule: assign each alternative to the highest category for which the alternative is at least as good as the boundary profile.


In [197]:
# CHECK EXACT
def calculate_pessimistic_assigment(relation: pd.DataFrame) -> pd.DataFrame:
    """
    Function that calculates pessimistic assigment for given relation between alternatives and boundary profiles

    :param relation: pandas dataframe with relation between alternatives as rows and boundary profiles as columns. With "<" or ">" for preference, "I" for indifference and "?" for incompatibility
    :return: dataframe with pessimistic assigment
    """
    num_alternatives = relation.shape[0]
    num_boundary_profiles = relation.shape[1]

    # Categories are defined by boundary profiles: b1..bm -> C1..C(m+1)
    categories = [f"C{i}" for i in range(1, num_boundary_profiles + 2)]
    assignment = pd.DataFrame(index=relation.index, columns=["Pessimistic Assignment"])

    for i in range(num_alternatives):
        assigned_category = categories[0]
        # Go from right: skip '<' and '?', stop at 'I' or '>'
        for j in range(num_boundary_profiles - 1, -1, -1):
            symbol = relation.iloc[i, j]
            if symbol in ["I", ">"]:
                assigned_category = categories[j + 1]
                break
        assignment.iloc[i, 0] = assigned_category

    return assignment

### Task 10 (maximum points: 1)

Implement the `calculate_optimistic_assigment` function.

Use the optimistic rule: assign each alternative to the lowest category whose upper boundary profile does not sufficiently outrank the alternative.


In [198]:
# TODO
# CHECK EXACT
def calculate_optimistic_assigment(relation: pd.DataFrame) -> pd.DataFrame:
    """
    Function that calculates optimistic assigment for given relation between alternatives and boundary profiles

    :param relation: pandas dataframe with relation between alternatives as rows and boundary profiles as columns. With "<" or ">" for preference, "I" for indifference and "?" for incompatibility
    :return: dataframe with optimistic assigment
    """
    num_alternatives = relation.shape[0]
    num_boundary_profiles = relation.shape[1]

    # Categories are defined by boundary profiles: b1..bm -> C1..C(m+1)
    categories = [f"C{i}" for i in range(1, num_boundary_profiles + 2)]
    assignment = pd.DataFrame(index=relation.index, columns=["Optimistic Assignment"])

    for i in range(num_alternatives):
        assigned_category = categories[-1]
        # Go from left: skip '>', 'I' and '?', stop at '<'
        for j in range(num_boundary_profiles):
            symbol = relation.iloc[i, j]
            if symbol == "<":
                assigned_category = categories[j]
                break
        assignment.iloc[i, 0] = assigned_category

    return assignment

In [199]:
marginal_concordance_matrix = calculate_marginal_concordance_matrix(
    dataset, boundary_profiles, preference_information
)
comprehensive_concordance_index = calculate_comprehensive_concordance_matrix(
    marginal_concordance_matrix, preference_information
)
marginal_discordance_matrix = calculate_marginal_discordance_matrix(
    dataset,
    boundary_profiles,
    preference_information,
    preference_information,
    preference_information,
)
credibility_index = calculate_credibility_index(
    comprehensive_concordance_index, marginal_discordance_matrix
)
outranking_relation_matrix = calculate_outranking_relation_matrix(
    credibility_index, credibility_threshold
)

In [200]:
marginal_concordance_matrix = calculate_marginal_concordance_matrix(
    dataset, boundary_profiles, preference_information
)
comprehensive_concordance_index = calculate_comprehensive_concordance_matrix(
    marginal_concordance_matrix, preference_information
)
marginal_discordance_matrix = calculate_marginal_discordance_matrix(
    dataset,
    boundary_profiles,
    preference_information,
    preference_information,
    preference_information,
)
credibility_index = calculate_credibility_index(
    comprehensive_concordance_index, marginal_discordance_matrix
)
outranking_relation_matrix = calculate_outranking_relation_matrix(
    credibility_index, credibility_threshold
)

In [201]:
relation = pd.DataFrame(index=dataset.index, columns=boundary_profiles.index)
for i, alt in enumerate(dataset.index):
    for j, profile in enumerate(boundary_profiles.index):
        a_outranks_b = outranking_relation_matrix[0, i, j]
        b_outranks_a = outranking_relation_matrix[1, i, j]
        if a_outranks_b and b_outranks_a:
            relation.loc[alt, profile] = "I"
        elif a_outranks_b and not b_outranks_a:
            relation.loc[alt, profile] = ">"
        elif not a_outranks_b and b_outranks_a:
            relation.loc[alt, profile] = "<"
        else:
            relation.loc[alt, profile] = "?"

In [202]:
pessimistic_assignment = calculate_pessimistic_assigment(relation)
optimistic_assignment = calculate_optimistic_assigment(relation)

results = pd.concat([relation, pessimistic_assignment, optimistic_assignment], axis=1)

In [203]:
print("=== Preference Information ===")
print(f"\nCredibility threshold (λ): {credibility_threshold}")
print(f"\nBoundary profiles (separate C1/C2/C3 classes):")
display(boundary_profiles)
print("\nPreference information (shared for all profiles):")
display(preference_information)
print("\nInterpretation:")
print("- C1 (weak): below b1")
print("- C2 (average): between b1 and b2")
print("- C3 (good): above b2")

=== Preference Information ===

Credibility threshold (λ): 0.7

Boundary profiles (separate C1/C2/C3 classes):


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions
Boundary,,,,,
b1,15,78,7,72,1
b2,22,88,8,86,3



Preference information (shared for all profiles):


,q,p,v,w,type
Criterion,,,,,
Hours_Studied,1,4,15.0,5,gain
Attendance,2,8,20.0,4,gain
Sleep_Hours,0,1,NaN,1,gain
Previous_Scores,3,10,20.0,2,gain
Tutoring_Sessions,0,1,NaN,3,gain



Interpretation:
- C1 (weak): below b1
- C2 (average): between b1 and b2
- C3 (good): above b2


In [204]:
print("=== Best and Worst Alternatives ===")

cat_map = {"C1": 1, "C2": 2, "C3": 3}
cred_a_b = pd.DataFrame(
    credibility_index[0], index=dataset.index, columns=boundary_profiles.index
).round(3)
cred_b_a = pd.DataFrame(
    credibility_index[1], index=dataset.index, columns=boundary_profiles.index
).round(3)

for method, assignment, col in [
    ("Pessimistic", pessimistic_assignment, "Pessimistic Assignment"),
    ("Optimistic", optimistic_assignment, "Optimistic Assignment"),
]:
    print(f"\n{'='*40}")
    print(f"{method} Assignment")
    print(f"{'='*40}")

    best_class = assignment[col].max()
    worst_class = assignment[col].min()

    best_alternatives = assignment[assignment[col] == best_class].index.tolist()
    worst_alternatives = assignment[assignment[col] == worst_class].index.tolist()

    print(f"\nBest class: {best_class} — alternatives: {best_alternatives}")
    display(dataset.loc[best_alternatives])
    if len(best_alternatives) > 1:
        print(f"Comparing within {best_class} using σ(a, b2):")
        display(cred_a_b.loc[best_alternatives])
        best_overall = cred_a_b.loc[best_alternatives, "b2"].idxmax()
        print(f"Best overall: {best_overall} — {dataset.loc[best_overall].to_dict()}")

    print(f"\nWorst class: {worst_class} — alternatives: {worst_alternatives}")
    display(dataset.loc[worst_alternatives])
    if len(worst_alternatives) > 1:
        print(f"Comparing within {worst_class} using σ(b1, a):")
        display(cred_b_a.loc[worst_alternatives])
        worst_overall = cred_b_a.loc[worst_alternatives, "b1"].idxmax()
        print(
            f"Worst overall: {worst_overall} — {dataset.loc[worst_overall].to_dict()}"
        )

print(f"\n{'='*40}")
print("Comparison between methods")
print(f"{'='*40}")
comparison = pd.concat(
    [
        pessimistic_assignment.rename(
            columns={"Pessimistic Assignment": "Pessimistic"}
        ),
        optimistic_assignment.rename(columns={"Optimistic Assignment": "Optimistic"}),
    ],
    axis=1,
)
comparison["Agree"] = comparison["Pessimistic"] == comparison["Optimistic"]
display(comparison)
print(f"\nAgreements: {comparison['Agree'].sum()}/{len(comparison)}")
print(
    f"Disagreements (incomparability): {(~comparison['Agree']).sum()}/{len(comparison)}"
)

=== Best and Worst Alternatives ===

Pessimistic Assignment

Best class: C3 — alternatives: ['Student_3', 'Student_4', 'Student_6']


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions
Student_3,24,98,7,91,2
Student_4,29,89,8,98,1
Student_6,19,88,8,89,3


Comparing within C3 using σ(a, b2):


Boundary,b1,b2
Student_3,1.0,0.733
Student_4,1.0,0.800
Student_6,1.0,0.778


Best overall: Student_4 — {'Hours_Studied': 29, 'Attendance': 89, 'Sleep_Hours': 8, 'Previous_Scores': 98, 'Tutoring_Sessions': 1}

Worst class: C1 — alternatives: ['Student_2', 'Student_8', 'Student_14', 'Student_15']


...

Best overall: Student_4 — {'Hours_Studied': 29, 'Attendance': 89, 'Sleep_Hours': 8, 'Previous_Scores': 98, 'Tutoring_Sessions': 1}

Worst class: C1 — alternatives: ['Student_15']


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions
Student_15,10,78,8,74,1



Comparison between methods


,Pessimistic,Optimistic,Agree
Student_1,C2,C2,True
Student_2,C1,C2,False
Student_3,C3,C3,True
Student_4,C3,C3,True
Student_5,C2,C2,True
Student_6,C3,C3,True
Student_7,C2,C3,False
Student_8,C1,C2,False
Student_9,C2,C2,True
Student_10,C2,C2,True



Agreements: 11/15
Disagreements (incomparability): 4/15


In [205]:
print("=== Pairwise Comparisons ===")
# Show credibility index for all pairs
print("\nCredibility σ(a, bh):")
cred_a_b = pd.DataFrame(
    credibility_index[0], index=dataset.index, columns=boundary_profiles.index
).round(3)
display(cred_a_b)

print("\nCredibility σ(bh, a):")
cred_b_a = pd.DataFrame(
    credibility_index[1], index=dataset.index, columns=boundary_profiles.index
).round(3)
display(cred_b_a)

print(f"\nOutranking holds when credibility >= λ={credibility_threshold}")
print("\nFinal relation matrix:")
display(relation)

=== Pairwise Comparisons ===

Credibility σ(a, bh):


Boundary,b1,b2
Student_1,0.800,0.511
Student_2,0.600,0.000
Student_3,1.000,0.733
Student_4,1.000,0.800
Student_5,0.857,0.000
Student_6,1.000,0.778
Student_7,0.981,0.209
Student_8,0.000,0.000
Student_9,0.733,0.343
Student_10,0.800,0.667



Credibility σ(bh, a):


Boundary,b1,b2
Student_1,0.489,1.000
Student_2,0.400,1.000
Student_3,0.000,0.584
Student_4,0.000,0.533
Student_5,0.125,0.911
Student_6,0.000,1.000
Student_7,0.087,0.667
Student_8,0.667,0.778
Student_9,0.371,0.822
Student_10,0.000,0.733



Outranking holds when credibility >= λ=0.7

Final relation matrix:


Boundary,b1,b2
Student_1,>,<
Student_2,?,<
Student_3,>,>
Student_4,>,>
Student_5,>,<
Student_6,>,I
Student_7,>,?
Student_8,?,<
Student_9,>,<
Student_10,>,<


In [206]:
print("=== Final Results ===")
final = dataset.copy()
final["Pessimistic"] = pessimistic_assignment["Pessimistic Assignment"]
final["Optimistic"] = optimistic_assignment["Optimistic Assignment"]
final["Agree"] = final["Pessimistic"] == final["Optimistic"]
display(final)

print("\nClass distribution (Pessimistic):")
print(pessimistic_assignment["Pessimistic Assignment"].value_counts().sort_index())
print("\nClass distribution (Optimistic):")
print(optimistic_assignment["Optimistic Assignment"].value_counts().sort_index())

print("\nAlternatives with different assignments (incomparability present):")
disagreements = final[~final["Agree"]][["Pessimistic", "Optimistic"]]
display(disagreements if len(disagreements) > 0 else "None — all assignments agree")

=== Final Results ===


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Pessimistic,Optimistic,Agree
Student_1,23,84,7,73,0,C2,C2,True
Student_2,19,64,8,59,2,C1,C2,False
Student_3,24,98,7,91,2,C3,C3,True
Student_4,29,89,8,98,1,C3,C3,True
Student_5,19,92,6,65,3,C2,C2,True
Student_6,19,88,8,89,3,C3,C3,True
Student_7,29,84,7,68,1,C2,C3,False
Student_8,25,78,6,50,1,C1,C2,False
Student_9,17,94,6,80,0,C2,C2,True
Student_10,23,98,8,71,0,C2,C2,True



Class distribution (Pessimistic):
Pessimistic Assignment
C1    4
C2    8
C3    3
Name: count, dtype: int64

Class distribution (Optimistic):
Optimistic Assignment
C1     1
C2    10
C3     4
Name: count, dtype: int64

Alternatives with different assignments (incomparability present):


,Pessimistic,Optimistic
Student_2,C1,C2
Student_7,C2,C3
Student_8,C1,C2
Student_14,C1,C2


## HOMEWORK

**Deadline:** 29.04.2026, 23:59

Using your implementations of the **PROMETHEE** and **ELECTRE** methods prepared during the laboratory sessions, perform an analysis of the dataset provided for the UTA laboratories.

As a solution, submit:

- Jupyter notebooks for:
  - PROMETHEE (max 7 points)
  - ELECTRE (max 10 points)
- A report in **PDF format**, in which you answer the questions listed below.

### PROMETHEE (max points: 10)

- Describe the preference information used as input to the method. _(2.0 points)_
- Present the final results obtained using the method. _(2.0 points)_
- Compare the complete and partial rankings. _(2.0 points)_
- Discuss your findings regarding the method, including:
  - the performance of the best and worst alternatives,
  - whether the pairwise comparisons defined in the dataset report are satisfied. _(4.0 points)_

### ELECTRE (max points: 10)

- Describe the preference information used as input to the method. _(2.0 points)_
- Present the final results obtained using the method. _(2.0 points)_
- Compare the optimistic and pessimistic class assignments. _(2.0 points)_
- Discuss your findings regarding the method, including:
  - the performance of the best and worst alternatives,
  - whether the pairwise comparisons defined in the dataset report are satisfied. _(4.0 points)_

### Method Comparison (max points: 3)

- Comment on the similarities and differences between the methods. _(3.0 points)_

**Final grade:**  
The final grade will be based on the total number of points obtained.
